# Longitudinal Analysis of the Allegheny County Jail Oversight Board Meeting Minutes

- Contributor: Danielle Aira Savellano
- AI Acknowledgements: The following code was written with assistance from GitHub Copilot.

## Text Preprocessing

### Text Extraction

Meeting minutes were downloaded from the [JOB website](https://alleghenycontroller.com/the-controller/jail-oversight-board/jail-oversight-board-internal/) as `PDF` or `DOCX` files for consistency in `0_0_File_Downloading.ipynb`.

This notebook contains the code used to:

- Convert documents into .TXT files.
- Save extracted text under `Data/Text/`
- Load .TXT files into Python

### Results

153 out of 156 files were successfully extracted into text. However, 3 files resulted in errors wherein blank text files were generated. Error Files were uploaded to a folder `Data/Error_Files/`. Upon inspection, errors were likely due to scanning quality.

Furthermore, some inconsistencies were identified, particularly in the more recent minutes. These minutes included glossaries, tables, and line numbering, which did not convert properly. With the focus on discussions, it was determined that the transcript text will be sufficient and so tables should not be a focus.

In [1]:
!pip install pypdf python-docx --quiet


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from pathlib import Path

# Paths
BASE = Path("..").resolve()
IN_DIR = BASE / "Data" / "Scraped"
OUT_DIR = BASE / "Data" / "Text"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Functions to read, extract, and convert files to plain text.

def read_text_file(path: Path) -> str:
    for encoding in ("utf-8", "utf-8-sig"):
        try:
            return path.read_text(encoding=encoding)
        except UnicodeDecodeError:
            continue
    return path.read_text(encoding="utf-8", errors="replace")

def extract_pdf_to_text(path: Path) -> str:
    '''Extract text from a PDF file using pypdf. Written by GitHub Copilot.'''
    from pypdf import PdfReader
    # strict=False tolerates slightly malformed PDFs from some scanners/tools.
    reader = PdfReader(str(path), strict=False)
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n\n".join(pages).strip()

def extract_docx_to_text(path: Path) -> str:
    '''Extract text from a docx file using python-docx. Written by GitHub Copilot.'''
    from docx import Document
    doc = Document(str(path))
    paragraphs = [p.text for p in doc.paragraphs]
    return "\n\n".join(paragraphs).strip()

def file_to_plain_text(path: Path) -> str:
    if path.suffix.lower() == ".pdf":
        return extract_pdf_to_text(path)
    if path.suffix.lower() == ".docx":
        return extract_docx_to_text(path)
    return read_text_file(path)

def list_input_files(directory: Path) -> list[Path]:
    # Sorted for reproducible order; skip dotfiles (.DS_Store, etc.).
    return sorted(
        p for p in directory.iterdir()
        if p.is_file() and not p.name.startswith(".")
    )

# Convert each input file to .txt for analysis and skip if output already exists
written: list[Path] = []
n_skipped = 0

for path in list_input_files(IN_DIR):
    # Skip non-document files (e.g., .csv)
    if path.suffix.lower() == ".csv":
        continue
    
    # Skip if the output .txt already exists (avoid re-processing)

    out_path = OUT_DIR / f"{path.stem}.txt"
    
    if out_path.exists() and out_path.stat().st_size == 0:  # Empty file exists
        n_skipped += 1
        written.append(out_path)

        print(f"Warning: {path.stem} already exists but is empty.")
        
        # Make copy of input file in Data/Error_Files/ for manual review if needed.
        error_dir = BASE / "Data" / "Error_Files"
        error_dir.mkdir(parents=True, exist_ok=True)
        error_path = error_dir / path.name
        if not error_path.exists():
            error_path.write_bytes(path.read_bytes())
        continue

    if out_path.exists() and out_path.stat().st_size > 0:
       n_skipped += 1
       written.append(out_path)
       continue                                            # Skip download

    text = file_to_plain_text(path)
    out_path.write_text(text, encoding="utf-8", newline="\n")
    written.append(out_path)

# Summary
print(f"\n{len(written)}/{len(list_input_files(IN_DIR))-2} Files Processed ({n_skipped} Already Existing)")
        
print(f"\nSaved to: {OUT_DIR}")


156/156 Files Processed (156 Already Existing)

Saved to: /Users/daesavellano/0_CMU/Unstructured Data Analytics/Project/board_meeting_analysis/Data/Text


In [3]:
# Load all converted .txt files for analysis

documents: list[str] = []                               # List of document texts
doc_names: list[str] = []                               # Corresponding list of filenames
for path in sorted(OUT_DIR.glob("*.txt")):
    documents.append(path.read_text(encoding="utf-8"))
    doc_names.append(path.name)

print(f"{len(documents)} document(s) → lists `documents` (text) and `doc_names` (filenames)")

156 document(s) → lists `documents` (text) and `doc_names` (filenames)
